## 1. What Will We Learn?

**The purpose of this notebook is to understand RAG, how it works, and where it is useful.**

RAG stands for Retrieval Augmented Generation. It combines a search system with a language model: the search finds relevant information, and the language model uses that information to write an answer.

We use song recommendations as a small, concrete example. A recommender first chooses songs. We then collect relevant descriptions and lyrics and ask Llama to explain possible connections with the songs the listener likes.

By the end, you should be able to explain how source text becomes searchable, how retrieved passages become part of a prompt, and how to check an answer against its sources. You should also understand when RAG is useful and what it cannot guarantee.

The song recommender provides the example; recommendation itself is not a required part of RAG. The same retrieval and generation steps can answer questions about manuals, policies, research papers, and other document collections.

## 2. What Is RAG?

Imagine asking an assistant a question about a document it has not been given. Its general knowledge may be insufficient or out of date. RAG addresses this by finding relevant source text and giving it to the model with the question.

There are two stages.

**Prepare the information.** Load documents, split them into smaller passages called chunks, and convert those chunks into embeddings. An embedding is a list of numbers that helps us compare text by meaning. Store the chunks, embeddings, and source details in a searchable collection.

**Answer a request.** Search for useful passages, put them in the prompt, and ask the language model to answer using them. This gives RAG its name:

1. **Retrieval:** Find relevant information.
2. **Augmentation:** Add that information to the model's prompt.
3. **Generation:** Produce an answer using the supplied context.

For example, a question about returning a product could retrieve the relevant paragraph from a store's return policy. The model can then explain the policy using that paragraph rather than guessing from general knowledge.

In this notebook, the request is to explain a song recommendation. We retrieve passages from the liked songs and the recommended song, add them to the prompt, and ask Llama to describe a supported connection. MiniLM creates embeddings, Chroma performs the search, and Llama writes the answer.

There are two searches here: one selects songs, while the other retrieves evidence for the explanation. Only the second search supplies the passages used in the RAG explanation. The selected songs are already fixed when Llama receives the prompt.

RAG does not retrain the language model. It provides information for the current answer. Updating the source collection can therefore change the evidence available to the model without changing its trained weights.

## 3. What Will We Compare?

To see the role of retrieved context, we ask Llama to explain each recommendation twice.

**Without retrieved passages:** Supply only the song IDs, titles, and genres.

**With retrieved passages:** Supply the same details plus relevant descriptions and lyrics.

We expect the supplied passages to make explanations more specific and easier to check. The comparison lets us inspect what changes when a model has source material available, rather than assuming RAG always improves an answer.

Each passage has a **chunk ID** so we can find its text and check the model's claims. This demonstrates the contribution of context; it does not establish that searching is better than supplying the entire small catalog.

## 4. Experiment Setup

We use eight historical songs with public domain lyrics. This small catalog is separate from the fictional catalog in the earlier notebooks and makes the evidence easy to inspect.

| Component | Role |
| --- | --- |
| `public-domain-songs.csv` | Song details, descriptions, source links, and rights notes |
| `public-domain-song-lyrics.csv` | Lyric sections linked to songs |
| `all-MiniLM-L6-v2` | Convert text into embeddings for search |
| Chroma | Store embeddings and find similar records |
| `llama3.2:3b` | Write explanations from the supplied records |

The data contains text, not audio. Descriptions were written for this experiment; source links identify the historical lyrics. Modern recordings and arrangements may have separate rights.

Run Llama locally using [Ollama](https://docs.ollama.com/quickstart). Download the model and keep Ollama running:

```sh
ollama pull llama3.2:3b
```

The next cell imports the libraries and sets the model and number of recommendations. We set temperature to `0` to reduce variation between answers, though this does not guarantee identical or correct output.

In [ ]:
import csv
import re
from dataclasses import dataclass
from pathlib import Path

import chromadb
from chromadb.utils import embedding_functions
from langchain_text_splitters import RecursiveCharacterTextSplitter
from ollama import Client

GENERATION_MODEL = "llama3.2:3b"
TEMPERATURE = 0
N_RECOMMENDATIONS = 3

### Load the catalog

Read the CSV rows into `Song` and `LyricSection` objects. One song can have several lyric sections, connected by `song_id`.

The code checks that every lyric section belongs to an existing song. The path fallback allows execution from either the repository root or this notebook's directory.

In [29]:
@dataclass(frozen=True)
class Song:
    song_id: str
    title: str
    artist: str
    genre: str
    description: str
    source_url: str
    rights_note: str

    def recommendation_text(self) -> str:
        return f"{self.title} {self.description}"


@dataclass(frozen=True)
class LyricSection:
    song_id: str
    section: str
    text: str


catalog_path = Path("public-domain-songs.csv")
lyrics_path = Path("public-domain-song-lyrics.csv")
if not catalog_path.exists():
    catalog_path = Path("embeddings-and-search") / catalog_path
    lyrics_path = Path("embeddings-and-search") / lyrics_path

with catalog_path.open(newline="", encoding="utf-8") as song_file:
    songs = [
        Song(
            song_id=row["id"],
            title=row["title"],
            artist=row["artist"],
            genre=row["genre"],
            description=row["description"],
            source_url=row["source_url"],
            rights_note=row["rights_note"],
        )
        for row in csv.DictReader(song_file)
    ]


with lyrics_path.open(newline="", encoding="utf-8") as lyrics_file:
    lyric_sections = [
        LyricSection(
            song_id=row["song_id"],
            section=row["section"],
            text=row["text"],
        )
        for row in csv.DictReader(lyrics_file)
    ]

songs_by_id = {song.song_id: song for song in songs}
unknown_lyric_ids = {section.song_id for section in lyric_sections} - songs_by_id.keys()
if unknown_lyric_ids:
    raise ValueError(f"Lyrics reference unknown songs: {sorted(unknown_lyric_ids)}")

print(f"Loaded {len(songs)} public-domain songs")
print(f"Loaded {len(lyric_sections)} lyric sections")

Loaded 8 public-domain songs
Loaded 27 lyric sections


### Chunk the explanation evidence

A **chunk** is a short passage used as evidence. Searching passages helps us select relevant text and cite its exact source.

The splitter tries paragraph, line, sentence, and space boundaries. Here, `chunk_size=180` limits chunks to about 180 characters, while `chunk_overlap=30` allows nearby chunks to share about 30 characters to preserve context. Short descriptions and lyric sections can remain intact.

Larger chunks preserve more context; smaller chunks focus the search. Overlap can protect a phrase at a boundary but also repeats text. These settings are teaching examples, not universal defaults.

Each chunk keeps its song ID, source type, section, URL, and rights note.

In [30]:
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". " , " "],
    chunk_size=180,
    chunk_overlap=30,
)

description_sources = [
    {
        "song": song,
        "source_type": "description",
        "section": "catalog_description",
        "text": song.description,
    }
    for song in songs
]

lyric_sources = [
    {
        "song": songs_by_id[lyric.song_id],
        "source_type": "lyrics",
        "section": lyric.section,
        "text": lyric.text,
    }
    for lyric in lyric_sections
]

evidence_sources = description_sources + lyric_sources

evidence_chunks = []
for source in evidence_sources:
    song = source["song"]
    for chunk_index, text in enumerate(splitter.split_text(source["text"])):
        evidence_chunks.append(
            {
                "id": (
                    f"{song.song_id}:{source['source_type']}:"
                    f"{source['section']}:chunk-{chunk_index}"
                ),
                "text": text,
                "metadata": {
                    "song_id": song.song_id,
                    "title": song.title,
                    "genre": song.genre,
                    "source_type": source["source_type"],
                    "section": source["section"],
                    "source_url": song.source_url,
                    "rights_note": song.rights_note,
                    "chunk_index": chunk_index,
                },
            }
        )

print(f"Created {len(evidence_chunks)} evidence chunks from {len(evidence_sources)} sources")
for chunk in evidence_chunks[:3]:
    print(f"\n{chunk['id']}: {chunk['text']}")

Created 65 evidence chunks from 35 sources

pd_001:description:catalog_description:chunk-0: A song of longing for an absent loved one, remembered through images of summer, flowing water, and a voice that is gone.

pd_002:description:catalog_description:chunk-0: A declaration of lasting devotion to Annie Laurie, whose remembered promise inspires complete loyalty.

pd_003:description:catalog_description:chunk-0: A promise that love will endure after youth and beauty fade, grounded in constancy rather than appearance.


### Build two indexes

| Collection | Stores | Used for |
| --- | --- | --- |
| Recommendation | One title and description document per song | Choosing similar songs |
| Evidence | Description and lyric chunks with source details | Gathering passages for explanations |

Keeping these separate means songs with more lyrics do not get extra entries in the recommendation search. MiniLM embeds the text in both collections.

`upsert` inserts new IDs or updates existing IDs, so rerunning the cell does not duplicate those records.

In [31]:
client = chromadb.Client()
embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

recommendation_collection = client.get_or_create_collection(
    name="public_domain_song_recommendations",
    embedding_function=embedding_function,
    configuration={"hnsw": {"space": "cosine"}},
)
recommendation_collection.upsert(
    ids=[song.song_id for song in songs],
    documents=[song.recommendation_text() for song in songs],
)

evidence_collection = client.get_or_create_collection(
    name="public_domain_song_evidence",
    embedding_function=embedding_function,
    configuration={"hnsw": {"space": "cosine"}},
)
evidence_collection.upsert(
    ids=[chunk["id"] for chunk in evidence_chunks],
    documents=[chunk["text"] for chunk in evidence_chunks],
    metadatas=[chunk["metadata"] for chunk in evidence_chunks],
)

print(f"Indexed {recommendation_collection.count()} songs")
print(f"Indexed {evidence_collection.count()} evidence chunks")

Indexed 8 songs
Indexed 65 evidence chunks


## 5. Retrieval: Find Candidate Songs

Before explaining anything, we need songs to explain. `recommend()` retrieves the embeddings of the liked songs and averages them into one vector representing the listener's interests. Chroma ranks catalog songs by similarity to that vector. The function excludes liked songs and returns the requested number of recommendations.

This is a comparison of text meanings, not a comparison of audio. An average can blur different interests, so it may return weak matches. Llama does not select or rerank the songs; the explanation stage should make a weak connection visible rather than disguise it.

In [ ]:
def recommend(liked: list[Song], n_results: int = 3) -> list[Song]:
    liked_ids = list(dict.fromkeys(song.song_id for song in liked))
    if not liked_ids or n_results <= 0:
        return []

    stored = recommendation_collection.get(ids=liked_ids, include=["embeddings"])
    embeddings = stored["embeddings"]
    if embeddings is None or len(embeddings) != len(liked_ids):
        raise ValueError("A liked song is missing an embedding")

    values_by_dimension = zip(*embeddings)

    # Average each dimension across all liked-song embeddings.
    average_embeddings = [
        sum(float(value) for value in dimension_values) / len(embeddings)
        for dimension_values in values_by_dimension
    ]
    results = recommendation_collection.query(
        query_embeddings=[average_embeddings],
        n_results=recommendation_collection.count(),
    )

    recommendations = [
        songs_by_id[song_id]
        for song_id in results["ids"][0]
        if song_id not in liked_ids
    ]
    return recommendations[:n_results]

In [ ]:
liked_songs = [songs_by_id[song_id] for song_id in ("pd_001", "pd_002", "pd_003")]
recommendations = recommend(liked_songs, N_RECOMMENDATIONS)

print("Liked songs:")
for song in liked_songs:
    print(f"- {song.song_id}: {song.title} [{song.genre}]\n  {song.description}")
    print(f"  Source: {song.source_url}")

print("\nRetrieved recommendations:")
for song in recommendations:
    print(f"- {song.song_id}: {song.title} [{song.genre}]\n  {song.description}")
    print(f"  Source: {song.source_url}")

## 6. Augmentation: Prepare the Evidence

The explanation needs support from both sides: passages showing a theme in the liked songs and passages showing whether the recommendation shares it. Without the recommended song's text, the model could describe the listener's interests without demonstrating that the new song relates to them.

`retrieve_evidence()` builds one search text from the recommended song's title and description and all the liked songs' descriptions. It then visits each involved song separately. Chroma uses MiniLM to embed the search text and compare it with the stored chunk embeddings.

The `song_id` filter means “search only this song.” With the default settings, each search returns up to two matching passages. The function puts each passage into a dictionary containing its text, chunk ID, source details, and role (`liked` or `recommended`), then returns the list.

This function gathers evidence; Llama later interprets the connection. Restricting searches to the involved songs ensures that each can contribute evidence, but the closest passage in a song can still be a poor match. There is no minimum similarity requirement here, and two passages may omit context or repeat overlapping text.

In [ ]:
def retrieve_evidence(
    liked: list[Song],
    recommended: Song,
    chunks_per_song: int = 2,
) -> list[dict]:
    liked_descriptions = " ".join(song.description for song in liked)
    question = (
        f"Recommended song: {recommended.title}. {recommended.description} "
        f"Liked-song themes: {liked_descriptions}"
    )
    records = []
    for song in [*liked, recommended]:
        result = evidence_collection.query(
            query_texts=[question],
            n_results=chunks_per_song,
            where={"song_id": song.song_id},
            include=["documents", "metadatas"],
        )
        for chunk_id, text, metadata in zip(
            result["ids"][0],
            result["documents"][0],
            result["metadatas"][0],
        ):
            records.append(
                {
                    "chunk_id": chunk_id,
                    "text": text,
                    "metadata": metadata,
                    "role": (
                        "recommended"
                        if song.song_id == recommended.song_id
                        else "liked"
                    ),
                }
            )
    return records

### Build the context and prompt

`build_metadata_context()` lists the involved songs and labels each as liked or recommended. `build_context()` adds the retrieved passages when evidence is provided.

`build_prompt()` combines that context with instructions to cite source IDs, avoid unsupported claims, and admit weak connections. This is the augmentation step: we give Llama the information it should use for this particular answer.

In [ ]:
def build_metadata_context(liked: list[Song], recommended: Song) -> str:
    songs_for_context = [*liked, recommended]
    return "\n".join(
        f"<song role='{('recommended' if song.song_id == recommended.song_id else 'liked')}' "
        f"id='{song.song_id}' title='{song.title}' genre='{song.genre}' />"
        for song in songs_for_context
    )


def build_context(
    liked: list[Song],
    recommended: Song,
    evidence: list[dict] | None = None,
) -> str:
    metadata_context = build_metadata_context(liked, recommended)
    if evidence is None:
        return metadata_context

    evidence_context = "\n\n".join(
        (
            f"<evidence role='{record['role']}' chunk_id='{record['chunk_id']}' "
            f"song_id='{record['metadata']['song_id']}' "
            f"source_type='{record['metadata']['source_type']}' "
            f"section='{record['metadata']['section']}' "
            f"source_url='{record['metadata']['source_url']}'>\n"
            f"{record['text']}\n"
            f"</evidence>"
        )
        for record in evidence
    )
    return f"{metadata_context}\n\n{evidence_context}"


def build_prompt(context: str, recommended: Song) -> str:
    return f"""
Explain why {recommended.title} ({recommended.song_id}) was recommended.

Use only the records in <context>.
- Write no more than three sentences.
- Cite supporting song IDs.
- Every claim based on a description or lyric must cite at least one exact chunk ID from the context.
- Do not invent audio qualities, listener preferences, or catalog facts.
- If the connection is weak or contradictory, say so explicitly.
- Do not promise that the listener will enjoy the song.

<context>
{context}
</context>
""".strip()

### Check citation IDs

A citation is useful only if we can find its source. `check_citations()` extracts IDs matching the notebook's citation pattern and checks them against the supplied song and chunk IDs. It also reports whether the answer contains a chunk citation.

This catches references to unknown IDs in that format. It does not verify quotations, check every claim for a citation, or determine whether a passage supports the sentence citing it. Those checks require reading the explanation alongside the evidence.

In [ ]:
CITATION_PATTERN = re.compile(r"\bpd_\d{3}(?::[A-Za-z0-9_-]+){0,3}\b")


def check_citations(
    explanation: str,
    liked: list[Song],
    recommended: Song,
    evidence: list[dict] | None = None,
) -> dict:
    evidence = evidence or []
    permitted_ids = {song.song_id for song in [*liked, recommended]}
    permitted_ids.update(record["chunk_id"] for record in evidence)
    cited_ids = set(CITATION_PATTERN.findall(explanation))
    cited_chunk_ids = {citation for citation in cited_ids if ":" in citation}
    return {
        "invalid_ids": sorted(cited_ids - permitted_ids),
        "has_chunk_citation": bool(cited_chunk_ids),
    }

In [35]:
example_evidence = retrieve_evidence(liked_songs, recommendations[0])
example_context = build_context(
    liked_songs, recommendations[0], evidence=example_evidence
)
example_prompt = build_prompt(example_context, recommendations[0])
print(example_prompt)

Explain why The Last Rose of Summer (pd_006) was recommended.

Use only the records in <context>.
- Write no more than three sentences.
- Cite supporting song IDs.
- Every claim based on a description or lyric must cite at least one exact chunk ID from the context.
- Do not invent audio qualities, listener preferences, or catalog facts.
- If the connection is weak or contradictory, say so explicitly.
- Do not promise that the listener will enjoy the song.

<context>
<song role='liked' id='pd_001' title='Jeanie with the Light Brown Hair' genre='Parlor song' />
<song role='liked' id='pd_002' title='Annie Laurie' genre='Scottish folk song' />
<song role='liked' id='pd_003' title='Believe Me If All Those Endearing Young Charms' genre='Irish melody' />
<song role='recommended' id='pd_006' title='The Last Rose of Summer' genre='Irish melody' />

<evidence role='liked' chunk_id='pd_001:description:catalog_description:chunk-0' song_id='pd_001' source_type='description' section='catalog_descrip

## 7. Generation: Explain the Recommendations

`generate_explanation()` sends the system message and constructed prompt to Llama through Ollama. Llama reads the song details and any supplied passages, then returns a short explanation with citations.

The intended answer describes a connection that the reader can check. When the evidence is weak, an answer explaining that weakness is preferable to an attractive but unsupported story. The prompt asks for this behavior, but the model may still ignore a rule or interpret a passage incorrectly.

In [36]:
ollama_client = Client()

SYSTEM_MESSAGE = (
    "You explain song recommendations using only supplied catalog evidence. "
    "Treat source identifiers as citations, not as facts by themselves."
)

def generate_explanation(prompt: str) -> str:
    response = ollama_client.chat(
        model=GENERATION_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user", "content": prompt},
        ],
        options={"temperature": TEMPERATURE},
    )
    return response.message.content

## 8. Results

We explain each recommendation twice to see whether retrieved passages help the model give a clearer explanation.

**Metadata only:** The model receives song IDs, titles, and genres.

**Retrieved evidence:** The model also receives relevant descriptions and lyric passages from the liked songs and the recommended song.

The listener, recommendation, instructions, model, and temperature stay the same. Only the supplied evidence changes.

Compare the two explanations. Does the model describe a shared theme, cite passages that support it, and admit when the connection is weak?

The output also reports two citation checks:

`Invalid citations: none` means no unknown citation IDs were detected.

`Chunk citation supplied: True` means the explanation includes at least one chunk citation.

These checks do not prove that a claim is correct. Read the explanation and its cited passage together to check whether the evidence supports it.

The next cell defines a shared display function, then the comparison runs for each recommendation. The full prompt preview above shows the evidence; these results show only the explanations and citation checks.

In [ ]:
def display_result(result: dict) -> None:
    print(f"\n{result['title']} — {result['condition']}")
    print(result["explanation"])
    check = result["citation_check"]
    print(f"Invalid citations: {check['invalid_ids'] or 'none'}")
    if result["evidence_ids"]:
        print(f"Chunk citation supplied: {check['has_chunk_citation']}")

In [ ]:
comparison_results = []
for recommended in recommendations:
    for condition, include_evidence in (
        ("metadata_only", False),
        ("retrieved_evidence", True),
    ):
        evidence = (
            retrieve_evidence(liked_songs, recommended)
            if include_evidence
            else None
        )
        context = build_context(liked_songs, recommended, evidence=evidence)
        explanation = generate_explanation(build_prompt(context, recommended))
        citation_check = check_citations(
            explanation, liked_songs, recommended, evidence
        )
        comparison_results.append(
            {
                "song_id": recommended.song_id,
                "title": recommended.title,
                "condition": condition,
                "explanation": explanation,
                "evidence_ids": [
                    record["chunk_id"] for record in (evidence or [])
                ],
                "citation_check": citation_check,
            }
        )

for result in comparison_results:
    display_result(result)

## 9. Benefits, Drawbacks, and Failure Cases

The main benefit of explanation RAG is **traceability**: the reader can inspect the passages behind an explanation instead of accepting a general statement about the songs. It also lets the model use the catalog's chosen text without relying on whether it encountered those sources during training.

That benefit depends on several things going right:

| Limitation | What can happen here |
| --- | --- |
| Missing context | Two chunks per song can leave out a verse that changes the interpretation. |
| Weak retrieval | A search returns the closest passages even when none provides a strong connection. |
| Selective evidence | A query built from the recommendation can favor matching themes and miss contradictions. |
| Model mistakes | Llama can misread a passage, invent a detail, or attach a valid citation to an unsupported claim. |
| Extra work | Chunking, embeddings, searches, and longer prompts add complexity and runtime. |
| Limited scope | Lyrics and descriptions cannot establish audio qualities or whether the listener will enjoy a recording. |

The lyric passages were not used to rank the recommendations, so they may support a plausible connection without explaining why the embedding search ranked a particular song highly.

Use this checklist to evaluate the answers:

| Check | Question |
| --- | --- |
| Specificity | Does the explanation identify a clear shared or conflicting theme? |
| Citation validity | Were the cited IDs supplied in the prompt? |
| Support | Do the actual passages support the claims and quotations? |
| Caution | Does it admit weak connections and avoid promising enjoyment? |

The next cell tries a listener with different interests. This explores less obvious connections, though it does not guarantee weak matches. The first recommendation is simply the closest available candidate, not necessarily a strong choice.

To audit a result, locate its chunk IDs in `evidence_chunks` and compare the text with each claim. Check evidence for both the liked songs and the recommended song. A fluent answer with valid IDs can still fail this review.

In [ ]:
mixed_likes = [songs_by_id[song_id] for song_id in ("pd_005", "pd_008")]
mixed_recommendations = recommend(mixed_likes, N_RECOMMENDATIONS)

print("Mixed listener recommendations:")
for recommended in mixed_recommendations:
    print(f"- {recommended.song_id}: {recommended.title} — {recommended.description}")

for recommended in mixed_recommendations:
    evidence = retrieve_evidence(mixed_likes, recommended)
    context = build_context(mixed_likes, recommended, evidence=evidence)
    explanation = generate_explanation(build_prompt(context, recommended))
    display_result(
        {
            "title": recommended.title,
            "condition": "retrieved_evidence",
            "explanation": explanation,
            "evidence_ids": [record["chunk_id"] for record in evidence],
            "citation_check": check_citations(
                explanation, mixed_likes, recommended, evidence
            ),
        }
    )

## 10. Where Can We Apply RAG?

The song example demonstrates a reusable process: prepare a collection of source passages, retrieve the passages relevant to a request, add them to a prompt, and generate an answer. In another application, the documents and questions change, but these stages remain.

| Application | What retrieval finds | What the model generates |
| --- | --- | --- |
| Internal knowledge assistant | Relevant policies and team documentation | An answer based on the organization's documents |
| Customer support | Product manuals and support articles | Instructions suited to the customer's question |
| Research assistant | Relevant passages from selected papers | A summary or comparison with source references |
| Technical documentation assistant | Documentation for a particular product version | An explanation based on that version |
| Recommendation explanation | Descriptions and passages for selected items | An explanation of a possible connection |

RAG is useful when answers depend on a document collection that is large, changes over time, or contains information the model may not know. Retrieval selects a manageable amount of source text for each request. Source IDs also give readers a way to inspect the evidence, although the application must preserve those references.

RAG adds work: sources must be prepared and maintained, retrieval takes time, and longer prompts add processing cost. A search can miss an important passage, and the model can misinterpret what it receives. In systems with private documents, retrieval must also respect which documents the user is allowed to access.

For a short document or this tiny song catalog, placing all relevant text directly in the prompt may be simpler. A direct database query is often better for an exact count or lookup. A task based only on general knowledge or creative writing may not need an external collection at all.

The central lesson is that retrieval supplies information, augmentation makes it available to the model, and generation turns it into an answer. Useful RAG requires both relevant sources and an answer that uses them accurately.

### Next Step: From Supplying Evidence to Learning a Task

In this notebook, we supplied Llama with relevant passages so it could explain a recommendation. We changed the information available in its prompt without changing the model itself.

Now imagine this system receiving messages from users. Some ask for recommendations, others want an explanation, and others report a missing playlist. Before retrieving evidence or writing an answer, the application needs to decide what each message is asking for.

This introduces a new task: **classifying intent using categories defined for our music service.** A message such as “Why does this song keep appearing?” might ask for a recommendation explanation or report a playback problem. Correct routing depends on the context and the category rules we define. More song lyrics alone will not resolve that distinction.

We will first try clear instructions and labelled examples in the prompt. If the model repeatedly confuses our categories, we can investigate **fine tuning**: additional training on examples of messages and their expected labels. Unlike supplying passages through RAG, fine tuning updates the model's trained parameters to encourage the desired classification behavior.

| Example message | Category | Route |
| --- | --- | --- |
| I like these songs. What else should I try? | Recommendation Request | Existing recommender, given liked song IDs |
| Why did you recommend this song? | Explanation Request | Existing RAG workflow, given the liked songs and recommendation |
| My playlist disappeared. | Customer Support | Support queue |
| Can we add our catalog to your service? | Partnership | Partnership queue |
| Buy followers through this link. | Junk | Junk queue |
| I have something to discuss. | Human Review | Human review queue |

Start with sample messages rather than a live inbox. Classification identifies intent; the application must separately obtain the song IDs and other details needed to handle the request. Searching directly from a free text theme would require an additional recommendation search function.

In the next notebook, define the category rules and build a labelled dataset, including ambiguous messages that should go to human review. Establish a baseline using prompting, then fine tune on the training examples. Compare both approaches on the same unseen test messages, keeping those messages out of training and prompt examples.

The learning question is: **Does fine tuning classify our messages more reliably than prompting alone?** Examine mistakes by category, especially useful messages incorrectly marked as junk, and whether unclear messages reach human review. Fine tuning adds data preparation and training work, so the comparison should establish whether it is worthwhile.

Fine tuning therefore extends this application rather than replacing RAG. The classifier learns how to route messages according to our examples and rules. When a message requests a recommendation explanation, the existing RAG workflow still retrieves the source passages needed to answer it.